# RQ3: How does framing emphasis vary across title and excerpt, and is it consistent?

In [25]:
import pandas as pd

df = pd.read_csv('full_data.csv')

In [26]:
import pandas as pd
from scipy.stats import chi2_contingency

title_col = "gpt_title_frame"
body_col = "gpt_excerpt_primary_frame"

frame_df = df.dropna(
    subset=[title_col, body_col]
).copy()

# Counts
frame_counts = pd.crosstab(
    frame_df[title_col],
    frame_df[body_col]
)

# Row percentages: excerpt distribution conditional on title frame
frame_pct = pd.crosstab(
    frame_df[title_col],
    frame_df[body_col],
    normalize="index"
).mul(100).round(1)

display(frame_pct)

gpt_excerpt_primary_frame,disruption,grievance,harm_to_protesters,impactful,peaceful,property_damage,protester_violence,residual,widespread_support
gpt_title_frame,,,,,,,,,
disruption,73.9,12.0,1.0,2.3,0.1,3.4,6.6,0.0,0.7
grievance,8.9,78.0,3.1,1.8,1.1,2.0,1.6,0.0,3.4
harm_to_protesters,5.4,0.7,87.8,0.4,0.0,0.4,4.7,0.2,0.4
impactful,7.4,10.4,3.8,70.4,0.2,0.9,5.2,0.0,1.6
peaceful,6.8,34.1,0.0,0.0,50.0,0.0,4.5,0.0,4.5
property_damage,3.3,4.4,0.6,1.7,0.0,75.7,13.8,0.6,0.0
protester_violence,1.4,0.8,2.1,0.8,0.3,5.9,88.7,0.0,0.0
residual,15.5,26.9,11.1,7.4,2.0,3.3,22.7,6.6,4.4
widespread_support,10.1,31.0,2.5,8.3,3.3,1.0,2.8,0.0,41.1


In [27]:
nonres = frame_df[
    frame_df[title_col] != "residual"
].copy()

# Does title frame equal excerpt primary frame?
nonres["same_frame"] = (
    nonres[title_col] == nonres[body_col]
)

reinforcement = (
    nonres.groupby(title_col)
    .agg(
        N=(body_col, "size"),
        same_frame_rate=("same_frame", "mean")
    )
)

reinforcement["same_frame_rate"] = (
    reinforcement["same_frame_rate"] * 100
).round(1)

reinforcement

,N,same_frame_rate
gpt_title_frame,,
disruption,865,73.9
grievance,930,78.0
harm_to_protesters,551,87.8
impactful,443,70.4
peaceful,44,50.0
property_damage,181,75.7
protester_violence,628,88.7
widespread_support,397,41.1


Both Harm To Protesters and Protester Violence show extremely high rates of having same title and excerpt frame. I.e. if the excerpt frames protesters as violent or victims of violence, this will communicated in the headline. Sensationalism/clickbait: people believe extremism will get views

In [28]:
from sklearn.metrics import cohen_kappa_score

kappa_all = cohen_kappa_score(
    frame_df[title_col],
    frame_df[body_col]
)

kappa_nonres = cohen_kappa_score(
    nonres[title_col],
    nonres[body_col]
)

print(f"All articles: {kappa_all:.3f}")
print(f"Non-Residual titles: {kappa_nonres:.3f}")

All articles: 0.300
Non-Residual titles: 0.702


In [29]:
df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length', 'gpt_title_frame', 'gpt_excerpt_primary_frame',
       'gpt_excerpt_secondary_frame', 'gpt_demands_present',
       'gpt_protester_quote', 'gpt_authority_quote', 'gpt_thematic_present',
       'frames_error', 'binary_error'],
      dtype='str')

In [30]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd

title_col = "gpt_title_frame"
body_col = "gpt_excerpt_primary_frame"

temp = df[[title_col, body_col]].dropna().copy()

# Normalize labels, if needed
for col in [title_col, body_col]:
    temp[col] = temp[col].astype(str).str.strip().str.lower()

nonres = temp[temp[title_col] != "residual"].copy()

for name, subset in [("All", temp), ("Non-Residual", nonres)]:
    observed = (df[title_col] == df[body_col]).mean()

    kappa = cohen_kappa_score(
        subset[title_col],
        subset[body_col]
    )

    expected = (observed - kappa) / (1 - kappa)

    print(
        f"{name}: N={len(df)}, "
        f"Agreement={observed:.3f}, "
        f"Expected={expected:.3f}, "
        f"Kappa={kappa:.3f}"
    )

All: N=9283, Agreement=0.365, Expected=0.092, Kappa=0.300
Non-Residual: N=9283, Agreement=0.365, Expected=-1.133, Kappa=0.702


In [32]:
import pandas as pd

title_col = "gpt_title_frame"
primary_col = "gpt_excerpt_primary_frame"
secondary_col = "gpt_excerpt_secondary_frame"

temp = df[
    [title_col, primary_col, secondary_col]
].dropna(subset=[title_col, primary_col]).copy()

# Normalize labels
for col in [title_col, primary_col, secondary_col]:
    temp[col] = temp[col].astype("string").str.strip().str.lower()

# Primary-only agreement
temp["match_primary"] = (
    temp[title_col] == temp[primary_col]
).fillna(False)

# Secondary-only agreement
temp["match_secondary"] = (
    temp[title_col] == temp[secondary_col]
).fillna(False)

# Match either primary or secondary
temp["match_either"] = (
    temp["match_primary"] | temp["match_secondary"]
)

# Secondary rescue: primary doesn't match, but secondary does
temp["secondary_rescue"] = (
    ~temp["match_primary"] & temp["match_secondary"]
)

# Compare all articles vs non-Residual titles
results = []

for name, df in [
    ("All articles", temp),
    ("Non-Residual titles", temp[temp[title_col] != "residual"])
]:
    results.append({
        "Sample": name,
        "N": len(df),
        "Primary match (%)": df["match_primary"].mean() * 100,
        "Primary or Secondary match (%)": df["match_either"].mean() * 100,
        "Secondary rescue (N)": df["secondary_rescue"].sum(),
        "Secondary rescue (%)": df["secondary_rescue"].mean() * 100
    })

agreement_summary = pd.DataFrame(results).round(1)

agreement_summary

,Sample,N,Primary match (%),Primary or Secondary match (%),Secondary rescue (N),Secondary rescue (%)
0,All articles,9283,36.5,43.3,637,6.9
1,Non-Residual titles,4039,75.2,91.0,637,15.8
